# Fine-Tuned Open-Source Price Prediction

Fine-tune **the same 3 open-source models** (gpt-oss-20b, Llama-3.2-3B, Qwen2.5-7B) with QLoRA on the Week 7 pricer data, then run the **same price prediction evaluation** (MAE, RMSLE, charts, comparison).

**Models (HuggingFace base → fine-tuned adapter):**
1. **openai/gpt-oss-20b**
2. **meta-llama/Llama-3.2-3B**
3. **Qwen/Qwen2.5-7B-Instruct**

**Prerequisites:** GPU (e.g. Colab A100), `HF_TOKEN`. For Llama and gpt-oss-20b, accept the model license on the Hugging Face hub. Run data + training cells first; then run evaluation cells. To fine-tune all 3 models, run the training cell three times with `MODEL_INDEX = 0`, then `1`, then `2`.

In [ ]:
!uv pip install --upgrade trl transformers

In [ ]:
# Optional: install deps for fine-tuning (uncomment if needed).
# PyTorch >= 2.4 is required (avoids "LRScheduler is not defined" when transformers disables torch on older PyTorch).
!uv pip install peft bitsandbytes accelerate datasets

In [ ]:
import os
import re
import math
import sys
from pathlib import Path

from dotenv import load_dotenv
from huggingface_hub import login
from tqdm.notebook import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, set_seed
from peft import LoraConfig, PeftModel
from trl import SFTTrainer, SFTConfig, DataCollatorForCompletionOnlyLM
from datasets import Dataset
import torch
import matplotlib.pyplot as plt
import pandas as pd

for d in [Path.cwd(), Path.cwd() / "week7"]:
    if d.exists() and (d / "pricer" / "items.py").exists():
        if str(d) not in sys.path:
            sys.path.insert(0, str(d))
        break

from pricer.items import Item

load_dotenv(override=True)
hf_token = os.environ.get("HF_TOKEN")
if hf_token:
    login(hf_token, add_to_git_credential=True)

## Data and prompts

In [ ]:
LITE_MODE = True
username = "ed-donner"
dataset = f"{username}/items_lite" if LITE_MODE else f"{username}/items_full"

train, val, test = Item.from_hub(dataset)
print(f"Loaded {len(train):,} train, {len(val):,} val, {len(test):,} test")

In [ ]:
BASE_MODEL_FOR_PROMPTS = "meta-llama/Llama-3.2-3B"
CUTOFF = 110

tokenizer_prompts = AutoTokenizer.from_pretrained(BASE_MODEL_FOR_PROMPTS)

for item in tqdm(train + val + test):
    item.make_prompts(tokenizer_prompts, CUTOFF, do_round=True)

def item_to_text(item: Item) -> str:
    return (item.prompt or "") + (item.completion or "")

train_texts = [item_to_text(x) for x in train]
val_texts = [item_to_text(x) for x in val]
train_dataset = Dataset.from_dict({"text": train_texts})
val_dataset = Dataset.from_dict({"text": val_texts})
print(f"Train samples: {len(train_dataset)}, Val samples: {len(val_dataset)}")

## Fine-tuning config: 3 models (gpt-oss-20b, Llama-3.2-3B, Qwen2.5-7B)

In [ ]:
MODELS_TO_FINETUNE = [
    {"name": "gpt-oss-20b", "base": "openai/gpt-oss-20b", "adapter_dir": "adapters/pricer-gpt-oss-20b"},
    {"name": "llama3.2-3b", "base": "meta-llama/Llama-3.2-3B", "adapter_dir": "adapters/pricer-llama32-3b"},
    {"name": "qwen2.5-7b", "base": "Qwen/Qwen2.5-7B-Instruct", "adapter_dir": "adapters/pricer-qwen25-7b"},
]

TRAIN_SUBSET = 2000   # Use first N samples for faster training; set to None for full train
MAX_SEQ_LENGTH = 182
RESPONSE_TEMPLATE = "Price is $"

## QLoRA training (run for each model you want to fine-tune)

In [ ]:
def train_one_model(model_config: dict, train_ds: Dataset, val_ds: Dataset):
    base_id = model_config["base"]
    adapter_dir = model_config["adapter_dir"]
    Path(adapter_dir).mkdir(parents=True, exist_ok=True)

    tokenizer = AutoTokenizer.from_pretrained(base_id, trust_remote_code=True)
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_quant_type="nf4",
    )
    model = AutoModelForCausalLM.from_pretrained(
        base_id,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True,
    )
    model.generation_config.pad_token_id = tokenizer.pad_token_id

    collator = DataCollatorForCompletionOnlyLM(RESPONSE_TEMPLATE, tokenizer=tokenizer)
    lora_config = LoraConfig(
        r=16,
        lora_alpha=32,
        target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
        lora_dropout=0.1,
        bias="none",
        task_type="CAUSAL_LM",
    )
    sft_config = SFTConfig(
        output_dir=adapter_dir,
        num_train_epochs=1,
        per_device_train_batch_size=4,
        gradient_accumulation_steps=2,
        learning_rate=5e-5,
        lr_scheduler_type="cosine",
        warmup_ratio=0.03,
        max_seq_length=MAX_SEQ_LENGTH,
        dataset_text_field="text",
        bf16=True,
        max_grad_norm=0.3,
        logging_steps=20,
        save_steps=500,
        save_total_limit=2,
        report_to="none",
    )
    trainer = SFTTrainer(
        model=model,
        args=sft_config,
        train_dataset=train_ds,
        peft_config=lora_config,
        data_collator=collator,
    )
    trainer.train()
    trainer.save_model(adapter_dir)
    tokenizer.save_pretrained(adapter_dir)
    print(f"Saved adapter to {adapter_dir}")
    return adapter_dir

In [ ]:
# Pick which model to fine-tune (0, 1, or 2); run this cell once per model or loop.
MODEL_INDEX = 0  # 0=gpt-oss-20b, 1=llama3.2-3b, 2=qwen2.5-7b

train_subset = train_dataset.select(range(min(TRAIN_SUBSET, len(train_dataset)))) if TRAIN_SUBSET else train_dataset
train_one_model(MODELS_TO_FINETUNE[MODEL_INDEX], train_subset, val_dataset)

## Prepare test prompts (for evaluation)

In [ ]:
# Test items already have .prompt/.completion from make_prompts; we need .test_prompt() for inference.
EVAL_SIZE = 100

## Inference and evaluation

In [ ]:
def extract_price(text: str):
    if not text or not text.strip():
        return None
    cleaned = text.replace(",", "").strip()
    match = re.search(r"\d+(?:\.\d+)?", cleaned)
    return float(match.group(0)) if match else None


def predict_with_finetuned(item: Item, model, tokenizer, device: str = "cuda"):
    set_seed(42)
    prompt = item.test_prompt()
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=20,
            pad_token_id=tokenizer.pad_token_id,
            do_sample=False,
        )
    full = tokenizer.decode(out[0], skip_special_tokens=True)
    # Completion is after "Price is $"
    if "Price is $" in full:
        tail = full.split("Price is $", 1)[1].strip()
        return extract_price(tail)
    return extract_price(full)

In [ ]:
def color_for(error: float, truth: float) -> str:
    if error < 40 or (truth and error / truth < 0.2):
        return "green"
    if error < 80 or (truth and error / truth < 0.4):
        return "orange"
    return "red"


def evaluate_finetuned_model(model_name: str, model, tokenizer, test_items: list, size: int = 100, device: str = "cuda"):
    guesses, truths, errors, sles, colors = [], [], [], [], []
    for i in tqdm(range(min(size, len(test_items))), desc=model_name):
        item = test_items[i]
        guess = predict_with_finetuned(item, model, tokenizer, device)
        truth = item.price
        guess = max(0.0, guess) if guess is not None else 0.0
        err = abs(guess - truth)
        log_err = math.log(truth + 1) - math.log(guess + 1)
        sle = log_err ** 2
        guesses.append(guess)
        truths.append(truth)
        errors.append(err)
        sles.append(sle)
        colors.append(color_for(err, truth))
    return guesses, truths, errors, sles, colors


def chart_predictions(truths, guesses, colors, title: str):
    plt.figure(figsize=(10, 8))
    max_val = max(max(truths), max(guesses), 1)
    plt.plot([0, max_val], [0, max_val], color="deepskyblue", lw=2, alpha=0.6)
    plt.scatter(truths, guesses, s=8, c=colors)
    plt.xlabel("Ground truth ($)")
    plt.ylabel("Model estimate ($)")
    plt.xlim(0, max_val)
    plt.ylim(0, max_val)
    plt.title(title)
    from matplotlib.lines import Line2D
    plt.legend(handles=[
        Line2D([0], [0], marker="o", color="w", label="Accurate (green)", markerfacecolor="green", markersize=8),
        Line2D([0], [0], marker="o", color="w", label="Medium (orange)", markerfacecolor="orange", markersize=8),
        Line2D([0], [0], marker="o", color="w", label="High error (red)", markerfacecolor="red", markersize=8),
    ], loc="upper right")
    plt.show()

In [ ]:
def load_finetuned_model(base_id: str, adapter_dir: str, device: str = "cuda"):
    tokenizer = AutoTokenizer.from_pretrained(adapter_dir, trust_remote_code=True)
    tokenizer.pad_token = tokenizer.eos_token
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_quant_type="nf4",
    )
    base_model = AutoModelForCausalLM.from_pretrained(
        base_id,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True,
    )
    model = PeftModel.from_pretrained(base_model, adapter_dir)
    model.eval()
    return model, tokenizer

In [ ]:
results_ft = {}
device = "cuda" if torch.cuda.is_available() else "cpu"

for cfg in MODELS_TO_FINETUNE:
    adapter_dir = cfg["adapter_dir"]
    if not Path(adapter_dir).exists() or not list(Path(adapter_dir).glob("adapter_*.safetensors")):
        print(f"Skipping {cfg['name']}: no adapter at {adapter_dir}. Run training first.")
        continue
    print(f"Loading {cfg['name']} ...")
    model, tokenizer = load_finetuned_model(cfg["base"], adapter_dir, device)
    guesses, truths, errors, sles, colors = evaluate_finetuned_model(
        cfg["name"], model, tokenizer, test, size=EVAL_SIZE, device=device
    )
    mae = sum(errors) / len(errors)
    rmsle = math.sqrt(sum(sles) / len(sles))
    hits = sum(1 for c in colors if c == "green")
    results_ft[cfg["name"]] = {"mae": mae, "rmsle": rmsle, "hits_pct": 100 * hits / len(colors)}
    print(f"MAE=${mae:,.2f}  RMSLE={rmsle:,.2f}  Green={hits}/{len(colors)} ({results_ft[cfg['name']]['hits_pct']:.1f}%)")
    chart_predictions(truths, guesses, colors, f"Fine-tuned {cfg['name']}  MAE=${mae:,.2f}  RMSLE={rmsle:,.2f}")
    del model
    torch.cuda.empty_cache() if device == "cuda" else None

## Comparison table (fine-tuned models)

In [ ]:
if results_ft:
    comparison_ft = pd.DataFrame(results_ft).T.round(2)
    comparison_ft.index.name = "model"
    comparison_ft
else:
    print("No fine-tuned adapters found. Run training cells first.")